<a href="https://colab.research.google.com/github/ghirr/Smart-Traffic-Flow-Prediction-and-Optimization/blob/main/Part2_Ensemble_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Partie 2 : Ensemble Learning, Bagging et Boosting

**Projet : Smart Traffic Flow Prediction and Optimization**

Cette partie utilise les jeux préparés par la partie 1 (`data_prepared/`, découpage chronologique train / validation / test, aucune fuite) et compare :

| Famille | Algorithme | Rôle |
|---|---|---|
| Référence | arbre de décision seul | montrer le surapprentissage d'un modèle unique |
| **Bagging** | **Random Forest** | réduire la variance |
| **Boosting** | **XGBoost** et AdaBoost | réduire le biais |

**Trois tâches** (toutes pour la **minute suivante**, par cellule de 100 m) :

| Tâche | Type | Cible | Métriques |
|---|---|---|---|
| `etat` | classification (3 classes) | congestionné / ralenti / fluide | accuracy, F1 macro, rappel par classe, AUC |
| `vitesse` | régression | vitesse moyenne `speed_next` | MAE, RMSE, R² |
| `flux` | régression | nombre de véhicules `n_veh_next` | MAE, RMSE, R² |

**Règle d'or** : les modèles sont ajustés sur le **train**, les hyperparamètres choisis sur la **validation**, et le **test** n'est utilisé qu'une seule fois, à la fin, pour la comparaison finale.

**Plan** : 1. données et références · 2. arbre seul · 3. Bagging · 4. Boosting · 5. comparaison · 6. évaluation finale sur le test · 7. avantages et limites.

In [ ]:
import os, sys, json, time, warnings, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
SEED = 42
np.random.seed(SEED)

# ---- Données préparées par la partie 1 (dossier data_prepared/ du dépôt) ----
REPO_URL = "https://github.com/ghirr/Smart-Traffic-Flow-Prediction-and-Optimization.git"
BRANCH   = "main"                      # branche du dépôt qui contient data_prepared/ et results/
if not os.path.exists("data_prepared/meta.json") and "google.colab" in sys.modules:
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, "/content/repo"], check=True)
    os.chdir("/content/repo")
assert os.path.exists("data_prepared/meta.json"), "Dossier data_prepared/ introuvable : exécutez la partie 1 ou clonez le dépôt."
os.makedirs("results", exist_ok=True)

META   = json.load(open("data_prepared/meta.json"))
FEATS  = META["features"]
ETATS  = META["etats"]                 # ['congestionné', 'ralenti', 'fluide']
SEUILS = META["seuils_etat_m_s"]
SPLITS = ("train", "val", "test")

def charger(suffixe):
    """Retourne les trois jeux (train, val, test) : suffixe = 'raw' (arbres) ou 'scaled' (ANN)."""
    return [pd.read_parquet(f"data_prepared/{s}_{suffixe}.parquet") for s in SPLITS]

# Les trois tâches de prédiction (cibles de la minute suivante)
TASKS = {
    "etat":    dict(target="etat_next_code", kind="clf", nom="État du trafic"),
    "vitesse": dict(target="speed_next",     kind="reg", nom="Vitesse moyenne (m/s)"),
    "flux":    dict(target="n_veh_next",     kind="reg", nom="Flux (nb de véhicules)"),
}

from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score, accuracy_score,
                             f1_score, recall_score, roc_auc_score, confusion_matrix)

def metriques(kind, y, pred, proba=None):
    """Métriques adaptées à la tâche : régression (MAE, RMSE, R²) ou classification (accuracy, F1 macro, rappels, AUC)."""
    if kind == "reg":
        return {"MAE": mean_absolute_error(y, pred), "RMSE": mean_squared_error(y, pred) ** 0.5, "R2": r2_score(y, pred)}
    rec = recall_score(y, pred, average=None, labels=[0, 1, 2])
    out = {"Accuracy": accuracy_score(y, pred), "F1_macro": f1_score(y, pred, average="macro"),
           "Rappel_congestionné": rec[0], "Rappel_ralenti": rec[1], "Rappel_fluide": rec[2]}
    if proba is not None:
        out["AUC_ovr"] = roc_auc_score(y, proba, multi_class="ovr")
    return out

def perte(kind, y, pred):
    """Critère unique de sélection (à minimiser) : MAE en régression, 1 - F1 macro en classification."""
    return mean_absolute_error(y, pred) if kind == "reg" else 1 - f1_score(y, pred, average="macro")

import xgboost as xgb
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import (RandomForestRegressor, RandomForestClassifier,
                              AdaBoostRegressor, AdaBoostClassifier)
from sklearn.inspection import permutation_importance
from sklearn.model_selection import ParameterSampler

## 1. Données et modèles de référence

On charge les jeux **bruts** (non normalisés) : les modèles à base d'arbres ne dépendent pas de l'échelle des variables.

Rappel de la partie 1 : `speed_next` n'existe que si la cellule contient au moins un véhicule à `t+1`. Les tâches `vitesse` et `etat` utilisent donc seulement ces lignes ; la tâche `flux` utilise toutes les lignes.

In [ ]:
RAW = charger("raw")

def X_y(df, tache):
    d = df.dropna(subset=[tache["target"]])
    y = d[tache["target"]].astype(int) if tache["kind"] == "clf" else d[tache["target"]].astype(float)
    return d[FEATS], y

DATA = {t: {s: X_y(df, TASKS[t]) for s, df in zip(SPLITS, RAW)} for t in TASKS}
print(pd.DataFrame({t: {s: len(DATA[t][s][1]) for s in SPLITS} for t in TASKS}).T.to_string())
print(f"\n{len(FEATS)} variables explicatives")

### Références simples (à battre)

Un modèle de Machine Learning n'a de valeur que s'il fait **mieux que des règles triviales**. On compare à :

- **Persistance** : la minute prochaine ressemble à la minute actuelle (flux = `n_veh`, vitesse = vitesse moyenne récente, état = état correspondant à la vitesse récente) ;
- **Profil de la cellule** : vitesse habituelle de la cellule (`cell_speed_mean`, calculée sur le train).

In [ ]:
PROFIL_FLUX = RAW[0].groupby("cell_id")["n_veh_next"].mean()       # flux habituel par cellule (train)

def references(split):
    i = SPLITS.index(split)
    out = {}
    X, y = DATA["vitesse"][split]
    out[("vitesse", "Persistance")]     = metriques("reg", y, X.speed_roll3.fillna(X.cell_speed_mean))
    out[("vitesse", "Profil cellule")]  = metriques("reg", y, X.cell_speed_mean)
    X, y = DATA["flux"][split]
    cell = RAW[i].loc[X.index, "cell_id"]
    out[("flux", "Persistance")]        = metriques("reg", y, X.n_veh_roll3)
    out[("flux", "Profil cellule")]     = metriques("reg", y, cell.map(PROFIL_FLUX).fillna(PROFIL_FLUX.mean()))
    X, y = DATA["etat"][split]
    out[("etat", "Persistance")]        = metriques("clf", y, np.digitize(X.speed_roll3.fillna(X.cell_speed_mean), SEUILS))
    out[("etat", "Profil cellule")]     = metriques("clf", y, np.digitize(X.cell_speed_mean, SEUILS))
    return out

ref_val = references("val")
cols = {"etat": "F1_macro", "vitesse": "MAE", "flux": "MAE"}
print("Références sur la validation (critère principal : F1 macro pour 'etat', MAE sinon) :")
for (t, n), m in ref_val.items():
    print(f"  {t:8s} {n:15s} {cols[t]} = {m[cols[t]]:.3f}")

## 2. Un seul arbre de décision : le surapprentissage

Un arbre profond « apprend par cœur » le train. On le montre en faisant varier la profondeur (tâche `vitesse`) : l'erreur d'entraînement descend toujours, l'erreur de validation remonte au-delà d'une certaine profondeur. C'est exactement ce que les méthodes d'ensemble corrigent.

In [ ]:
Xa, ya = DATA["vitesse"]["train"]; Xb, yb = DATA["vitesse"]["val"]
prof = list(range(1, 26))
mae_tr, mae_va = [], []
for d in prof:
    arbre = DecisionTreeRegressor(max_depth=d, random_state=SEED).fit(Xa, ya)
    mae_tr.append(mean_absolute_error(ya, arbre.predict(Xa)))
    mae_va.append(mean_absolute_error(yb, arbre.predict(Xb)))

best_d = prof[int(np.argmin(mae_va))]
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(prof, mae_tr, "o-", label="train"); ax.plot(prof, mae_va, "o-", label="validation")
ax.axvline(best_d, color="gray", ls="--"); ax.set_xlabel("profondeur de l'arbre"); ax.set_ylabel("MAE (m/s)")
ax.set_title("Arbre seul : le surapprentissage"); ax.legend(); plt.show()
print(f"Meilleure profondeur (validation) : {best_d}, MAE val = {min(mae_va):.3f}")
print(f"Arbre non limité (profondeur 25) : MAE train = {mae_tr[-1]:.3f} vs MAE val = {mae_va[-1]:.3f}")
arbre_ref = {"depth": best_d, "mae_val": min(mae_va)}

## 3. Bagging : Random Forest

**Principe.**
1. On tire `B` échantillons **bootstrap** du train (tirage avec remise).
2. On entraîne un arbre profond sur chacun.
3. À chaque coupure, l'arbre ne regarde qu'un **sous-ensemble aléatoire de variables** (c'est ce qui distingue la forêt aléatoire du bagging simple).
4. Prédiction : **moyenne** des arbres (régression) ou **vote** (classification).

Chaque arbre seul a une forte variance (il suit le bruit de son échantillon) ; la moyenne de nombreux arbres peu corrélés a une variance bien plus faible, sans augmenter le biais.

**Pourquoi ce choix ici ?**
- Les variables sont nombreuses et **corrélées** (retards, moyennes glissantes) : la forêt le tolère bien.
- Les relations sont **non linéaires** et locales (zone lente / rapide, effets de seuil dus aux feux).
- Le jeu a des **valeurs extrêmes** (pics de flux) : les arbres y sont robustes, sans normalisation.
- Peu sensible au choix des hyperparamètres, c'est un bon point de départ.

**Hyperparamètres optimisés** (recherche aléatoire, évaluée sur la validation) : nombre d'arbres, profondeur maximale, taille minimale des feuilles, fraction de variables par coupure, taille des échantillons bootstrap.

In [ ]:
ESPACES = {
    "RandomForest": {"n_estimators": [100, 150], "max_depth": [12, 20, None],
                     "min_samples_leaf": [3, 5, 10], "max_features": ["sqrt", 0.3],
                     "max_samples": [0.5, 0.7]},
    "XGBoost":      {"learning_rate": [0.02, 0.05, 0.1], "max_depth": [3, 4, 6, 8], "min_child_weight": [1, 5, 10],
                     "subsample": [0.6, 0.8, 1.0], "colsample_bytree": [0.5, 0.7, 1.0], "reg_lambda": [0.5, 1, 5, 10]},
    "AdaBoost":     {"n_estimators": [50, 100, 200], "learning_rate": [0.05, 0.1, 0.5, 1.0], "depth": [2, 3, 4]},
}
N_ITER = {"RandomForest": 6, "XGBoost": 25, "AdaBoost": 4}

def fabrique(famille, kind, p):
    p = dict(p)
    if famille == "RandomForest":
        cls = RandomForestClassifier if kind == "clf" else RandomForestRegressor
        return cls(n_jobs=-1, random_state=SEED, **p)
    if famille == "XGBoost":
        cls = xgb.XGBClassifier if kind == "clf" else xgb.XGBRegressor
        return cls(n_estimators=2000, early_stopping_rounds=50, tree_method="hist", n_jobs=-1, random_state=SEED,
                   eval_metric="mlogloss" if kind == "clf" else "mae", **p)
    if famille == "AdaBoost":
        d = p.pop("depth")
        if kind == "clf":
            return AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=d, random_state=SEED), random_state=SEED, **p)
        return AdaBoostRegressor(estimator=DecisionTreeRegressor(max_depth=d, random_state=SEED), random_state=SEED, **p)

def ajuster(famille, modele, Xa, ya, Xb, yb):
    if famille == "XGBoost":      # arrêt anticipé : on garde le nombre d'arbres qui minimise l'erreur de validation
        modele.fit(Xa, ya, eval_set=[(Xa, ya), (Xb, yb)], verbose=False)
    else:
        modele.fit(Xa, ya)
    return modele

def chercher(famille, tache):
    """Recherche aléatoire : on entraîne sur le train, on évalue sur la validation, on garde le meilleur."""
    kind = TASKS[tache]["kind"]
    (Xa, ya), (Xb, yb) = DATA[tache]["train"], DATA[tache]["val"]
    essais, meilleur = [], (np.inf, None, None, None)
    for p in ParameterSampler(ESPACES[famille], N_ITER[famille], random_state=SEED):
        t0 = time.time()
        mod = ajuster(famille, fabrique(famille, kind, p), Xa, ya, Xb, yb)
        temps = time.time() - t0
        sc = perte(kind, yb, mod.predict(Xb))
        ligne = {**p, "perte_val": sc, "temps_s": temps}
        if famille == "XGBoost": ligne["n_arbres"] = int(mod.best_iteration + 1)
        essais.append(ligne)
        if sc < meilleur[0]: meilleur = (sc, p, mod, temps)
    return pd.DataFrame(essais).sort_values("perte_val").reset_index(drop=True), meilleur

MODELES, ESSAIS, PARAMS, TEMPS = {}, {}, {}, {}      # résultats par (tâche, famille)
def lancer(famille):
    for t in TASKS:
        essais, (sc, p, mod, temps) = chercher(famille, t)
        ESSAIS[(t, famille)], MODELES[(t, famille)], PARAMS[(t, famille)], TEMPS[(t, famille)] = essais, mod, p, temps
        print(f"{famille:12s} {t:8s}  meilleure perte validation = {sc:.4f}  ({len(essais)} essais, {essais.temps_s.sum():.0f} s)")

In [ ]:
lancer("RandomForest")

In [ ]:
for t in TASKS:
    e = ESSAIS[(t, "RandomForest")]
    print(f"\n=== RandomForest / {t} : 3 meilleurs essais (perte = {'1 - F1 macro' if TASKS[t]['kind']=='clf' else 'MAE'}) ===")
    print(e.head(3).round(4).to_string(index=False))

**Combien d'arbres faut-il ?** On ajoute des arbres un par un (à hyperparamètres fixés) et on regarde l'erreur de validation : elle se stabilise, **ajouter des arbres ne cause pas de surapprentissage** en bagging, il coûte seulement du temps de calcul.

In [ ]:
p = dict(PARAMS[("vitesse", "RandomForest")]); p.pop("n_estimators")
foret = RandomForestRegressor(n_estimators=10, warm_start=True, n_jobs=-1, random_state=SEED, **p)
Xa, ya = DATA["vitesse"]["train"]; Xb, yb = DATA["vitesse"]["val"]
courbe = []
for n in [10, 25, 50, 100, 150, 250]:
    foret.set_params(n_estimators=n).fit(Xa, ya)
    courbe.append((n, mean_absolute_error(yb, foret.predict(Xb))))
courbe = pd.DataFrame(courbe, columns=["n_arbres", "MAE_val"])
ax = courbe.plot(x="n_arbres", y="MAE_val", marker="o", figsize=(6.5, 3.6), legend=False)
ax.set_ylabel("MAE validation (m/s)"); ax.set_title("Random Forest : erreur de validation selon le nombre d'arbres"); plt.show()
print(courbe.round(4).to_string(index=False))

## 4. Boosting : XGBoost et AdaBoost

**Principe général.** Au lieu de moyenner des arbres indépendants, le boosting construit les arbres **l'un après l'autre**, chacun corrigeant les erreurs des précédents. Il réduit surtout le **biais** (des arbres simples et peu profonds deviennent, ensemble, très expressifs).

- **AdaBoost** : après chaque arbre, on **augmente le poids des exemples mal prédits** ; l'arbre suivant se concentre dessus. Les arbres sont pondérés dans le vote final selon leur précision. Sensible au bruit et aux valeurs aberrantes.
- **XGBoost** : *gradient boosting* ; chaque nouvel arbre est ajusté sur les **résidus** (le gradient de la perte) des arbres précédents. Il ajoute de la **régularisation** (`reg_lambda`, `min_child_weight`), du sous-échantillonnage des lignes (`subsample`) et des colonnes (`colsample_bytree`), et un **taux d'apprentissage** (`learning_rate`) qui fait avancer par petits pas.

**Pourquoi ce choix ici ?** Même raisons que pour la forêt (non-linéarités, variables corrélées, pas de normalisation), plus : le boosting donne en général la meilleure précision sur données tabulaires de cette taille, et XGBoost est rapide (histogrammes) et dispose de l'**arrêt anticipé** : on arrête quand l'erreur de validation cesse de baisser.

**Hyperparamètres optimisés.** XGBoost : `learning_rate`, `max_depth`, `min_child_weight`, `subsample`, `colsample_bytree`, `reg_lambda` (le nombre d'arbres est choisi automatiquement par arrêt anticipé). AdaBoost : `n_estimators`, `learning_rate`, profondeur des arbres faibles.

In [ ]:
lancer("XGBoost")

In [ ]:
for t in TASKS:
    print(f"\n=== XGBoost / {t} : 3 meilleurs essais ===")
    print(ESSAIS[(t, "XGBoost")].head(3).round(4).to_string(index=False))

In [ ]:
lancer("AdaBoost")
for t in TASKS:
    print(f"\n=== AdaBoost / {t} ===")
    print(ESSAIS[(t, "AdaBoost")].head(3).round(4).to_string(index=False))

### Courbe d'apprentissage du boosting

L'erreur d'**entraînement** baisse sans cesse, celle de **validation** baisse puis se stabilise : l'arrêt anticipé s'arrête à ce moment-là.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
for a, t in zip(ax, TASKS):
    mod = MODELES[(t, "XGBoost")]; h = mod.evals_result()
    cle = list(h["validation_0"].keys())[0]
    a.plot(h["validation_0"][cle], label="train"); a.plot(h["validation_1"][cle], label="validation")
    a.axvline(mod.best_iteration, color="gray", ls="--", label=f"arrêt ({mod.best_iteration + 1} arbres)")
    a.set_title(f"XGBoost / {t} ({cle})"); a.set_xlabel("nombre d'arbres"); a.legend()
plt.tight_layout(); plt.show()

## 5. Comparaison sur la validation

Tous les modèles sont comparés **sur la validation** (choix des hyperparamètres), avec leur temps d'entraînement et leur écart train / validation (indice de surapprentissage).

In [ ]:
FAMILLES = ["RandomForest", "XGBoost", "AdaBoost"]
def proba(modele, X):
    return modele.predict_proba(X) if hasattr(modele, "predict_proba") else None

lignes = []
for t in TASKS:
    kind = TASKS[t]["kind"]; crit = "F1_macro" if kind == "clf" else "MAE"
    for f in FAMILLES:
        mod = MODELES[(t, f)]
        (Xa, ya), (Xb, yb) = DATA[t]["train"], DATA[t]["val"]
        mv = metriques(kind, yb, mod.predict(Xb), proba(mod, Xb) if kind == "clf" else None)
        mt = metriques(kind, ya, mod.predict(Xa))
        lignes.append({"tâche": t, "modèle": f, **{f"{k}_val": v for k, v in mv.items()},
                       f"{crit}_train": mt[crit], "écart_train_val": abs(mt[crit] - mv[crit]), "temps_s": TEMPS[(t, f)]})
for (t, n), m in ref_val.items():
    kind = TASKS[t]["kind"]
    lignes.append({"tâche": t, "modèle": f"[réf] {n}", **{f"{k}_val": v for k, v in m.items()}})
comp_val = pd.DataFrame(lignes)

for t in TASKS:
    kind = TASKS[t]["kind"]
    cols = ["modèle"] + (["Accuracy_val", "F1_macro_val", "Rappel_congestionné_val", "AUC_ovr_val"] if kind == "clf" else ["MAE_val", "RMSE_val", "R2_val"]) + ["écart_train_val", "temps_s"]
    print(f"\n=== {t} : {TASKS[t]['nom']} ===")
    print(comp_val[comp_val["tâche"] == t][cols].round(3).to_string(index=False))

## 6. Évaluation finale sur le **test**

On utilise maintenant le jeu de test, **une seule fois**, avec les modèles retenus sur la validation. Les prédictions sont sauvegardées dans `results/` pour la comparaison finale (partie 5).

In [ ]:
resultats, preds = {}, {}
for t in TASKS:
    kind = TASKS[t]["kind"]; Xc, yc = DATA[t]["test"]
    preds[t] = {"y_true": yc.values}
    resultats[t] = {}
    for f in FAMILLES:
        mod = MODELES[(t, f)]
        p = mod.predict(Xc); pr = proba(mod, Xc) if kind == "clf" else None
        resultats[t][f] = {"test": metriques(kind, yc, p, pr),
                           "val": metriques(kind, DATA[t]["val"][1], mod.predict(DATA[t]["val"][0]), proba(mod, DATA[t]["val"][0]) if kind == "clf" else None),
                           "temps_s": TEMPS[(t, f)], "params": {k: (None if v is None else v) for k, v in PARAMS[(t, f)].items()}}
        preds[t][f] = p

rows = []
for t in TASKS:
    crit = "F1_macro" if TASKS[t]["kind"] == "clf" else "MAE"
    for f in FAMILLES:
        r = resultats[t][f]; rows.append({"tâche": t, "modèle": f, f"{crit} val": r["val"][crit], f"{crit} test": r["test"][crit]})
print(pd.DataFrame(rows).round(3).to_string(index=False))

# Sauvegardes pour la partie 5
json.dump(resultats, open("results/ensemble_metrics.json", "w"), indent=2, default=lambda o: float(o) if hasattr(o, "__float__") else str(o))
for t in TASKS:
    pd.DataFrame(preds[t]).to_parquet(f"results/ensemble_preds_{t}.parquet", index=False)
print("\nSauvegardé : results/ensemble_metrics.json et results/ensemble_preds_*.parquet")

In [ ]:
# Détail pour la classification : matrice de confusion du meilleur modèle (test)
t = "etat"; Xc, yc = DATA[t]["test"]
meilleur = max(FAMILLES, key=lambda f: resultats[t][f]["test"]["F1_macro"])
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for a, norm in zip(ax, [None, "true"]):
    cm = confusion_matrix(yc, preds[t][meilleur], normalize=norm)
    sns.heatmap(cm, annot=True, fmt=".2f" if norm else "d", cmap="Blues", xticklabels=ETATS, yticklabels=ETATS, ax=a, cbar=False)
    a.set_xlabel("prédit"); a.set_ylabel("réel"); a.set_title(f"{meilleur} (test)" + (" : normalisée par classe réelle" if norm else ""))
plt.tight_layout(); plt.show()

### Variables les plus importantes (tâche `vitesse`)

Deux regards complémentaires :
- **Importance par permutation** (forêt) : de combien l'erreur de validation augmente quand on **mélange** une variable. Elle mesure ce dont le modèle se sert vraiment, pas une cause réelle.
- **Gain** (XGBoost) : la réduction d'erreur apportée par les coupures sur cette variable.

In [ ]:
Xb, yb = DATA["vitesse"]["val"]
perm = permutation_importance(MODELES[("vitesse", "RandomForest")], Xb, yb, n_repeats=3, random_state=SEED, n_jobs=-1,
                              scoring="neg_mean_absolute_error")
imp_rf = pd.Series(perm.importances_mean, index=FEATS).sort_values(ascending=False)
gain = pd.Series(MODELES[("vitesse", "XGBoost")].get_booster().get_score(importance_type="gain")).reindex(FEATS).fillna(0)
imp_xg = (gain / gain.sum()).sort_values(ascending=False)

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
imp_rf.head(12)[::-1].plot.barh(ax=ax[0], color="seagreen"); ax[0].set_title("Random Forest : importance par permutation\n(hausse de MAE si on mélange la variable)")
imp_xg.head(12)[::-1].plot.barh(ax=ax[1], color="steelblue"); ax[1].set_title("XGBoost : part du gain")
plt.tight_layout(); plt.show()
print("Top 5 forêt  :", list(imp_rf.index[:5]))
print("Top 5 XGBoost:", list(imp_xg.index[:5]))

## 7. Analyse : avantages et limites des approches

**Ce que montrent les résultats** (tous les chiffres viennent des cellules ci-dessus ; la validation a servi aux réglages, le test à la mesure finale).

1. **Les modèles d'ensemble battent nettement l'arbre seul.** L'arbre limité à la meilleure profondeur atteint une MAE de vitesse d'environ 1,33 m/s en validation ; non limité, il tombe à 0,33 m/s sur le train mais remonte à 1,72 sur la validation (surapprentissage net). La forêt et XGBoost descendent à environ 1,29 m/s.
2. **Mais l'apport par rapport aux références simples est modeste.** Le profil habituel d'une cellule donne déjà une MAE de vitesse de 1,36 m/s (validation) contre 1,29 pour les meilleurs modèles, et un F1 de 0,71 contre 0,74. Le trafic simulé est largement déterminé par la **géographie** : la variable `cell_speed_mean` domine de très loin l'importance, devant la vitesse récente. Les modèles apportent un gain réel, mais plus faible que ce que laisserait croire un score brut.
3. **Bagging contre Boosting : XGBoost et la forêt sont très proches.** XGBoost est légèrement devant sur la vitesse et le flux (test : 1,280 contre 1,285 m/s, et 1,445 contre 1,474 véhicule) et à égalité sur l'état du trafic (F1 de 0,730 pour les deux). Ces écarts sont petits : la partie 5 teste s'ils dépassent le bruit statistique.
4. **AdaBoost est nettement moins bon** (MAE de vitesse 1,47 contre 1,28) et c'est le plus lent à entraîner. Son poids sur les exemples mal prédits le rend sensible au bruit, et les trafics voisins sont intrinsèquement bruités (arrivées aléatoires, feux). Son écart train / validation est faible : il **sous-apprend**, avec la petite grille d'hyperparamètres explorée ici.
5. **Surapprentissage** : la forêt a le plus grand écart train / validation (0,29 m/s de MAE sur la vitesse) car ses arbres profonds mémorisent le train ; XGBoost, régularisé et arrêté tôt, a un écart plus faible (0,10).

**Avantages et limites**

| | Random Forest (Bagging) | XGBoost (Boosting) | AdaBoost |
|---|---|---|---|
| Précision ici | très bonne | meilleure ou égale | moyenne |
| Hyperparamètres | peu sensibles : de nombreux réglages donnent presque le même score | plus nombreux et plus sensibles, mais l'arrêt anticipé fixe seul le nombre d'arbres | sensible au taux d'apprentissage et à la profondeur |
| Surapprentissage | modéré (grand écart train / val sans effet sur la validation) | contrôlé par la régularisation et l'arrêt anticipé | faible, mais sous-apprentissage |
| Temps d'entraînement | quelques secondes (parallélisable) | quelques secondes avec arrêt anticipé | le plus long (arbres construits en série et recalculs de poids) |
| Robustesse au bruit et aux extrêmes | bonne (moyenne de nombreux arbres) | bonne avec régularisation | faible |
| Interprétabilité | importance par permutation | importance par gain | importance par impureté |

**Limites de cette étude.** (i) La recherche d'hyperparamètres est volontairement **restreinte** (6, 25 et 4 essais) pour un temps de calcul raisonnable ; avec plus d'essais, AdaBoost et la forêt pourraient gagner un peu. (ii) Le jeu de validation sert à la fois à choisir les réglages et à l'arrêt anticipé de XGBoost : le score de validation est donc légèrement optimiste, d'où l'écart observé avec le test pour le flux (1,38 en validation, 1,45 en test). (iii) Les erreurs ne sont pas indépendantes (cellules voisines, minutes consécutives) : c'est pourquoi la partie 5 utilise un bootstrap par blocs de minutes. (iv) Une seule simulation : on ne sait pas si les résultats se généralisent à un autre réseau ou à une autre période de la journée.